# Clear GDELT Data

This notebook collapses duplicate intra-day GDELT timestamps into one row per day by averaging numeric columns.

In [1]:
from pathlib import Path

import pandas as pd

INPUT_PATH = Path("../data/equity_data/gdelt_data.csv")
OUTPUT_PATH = Path("../data/equity_data/gdelt_data_daily.csv")

df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df.head()

,Date,gdelt_articles,gdelt_robust,sentiment_score
0,2023-01-01,0.3065,-0.224062,-0.9249
1,2023-01-02,0.4465,0.548565,-0.7794
2,2023-01-03,0.8445,2.745033,-0.9603
3,2023-01-04,0.8318,2.674945,-1.7953
4,2023-01-05,0.5053,0.873068,-1.3037


In [2]:
df["day"] = df["Date"].dt.normalize()
duplicate_rows = df[df["day"].duplicated(keep=False)].copy()

print(f"Rows before cleaning: {len(df)}")
print(f"Rows with duplicate day values: {len(duplicate_rows)}")
duplicate_rows.head(20)

Rows before cleaning: 1309
Rows with duplicate day values: 240


,Date,gdelt_articles,gdelt_robust,sentiment_score,day
1067,2025-12-21 00:00:00,0.3636,0.091060,1.2425,2025-12-21
1068,2025-12-21 01:00:00,0.2223,-0.688742,-0.0738,2025-12-21
1069,2025-12-21 02:00:00,0.4289,0.451435,1.3889,2025-12-21
1070,2025-12-21 03:00:00,0.2528,-0.520419,1.4672,2025-12-21
1071,2025-12-21 04:00:00,0.4831,0.750552,0.1232,2025-12-21
1072,2025-12-21 05:00:00,0.3144,-0.180464,0.3356,2025-12-21
1073,2025-12-21 06:00:00,0.1975,-0.825607,0.2909,2025-12-21
1074,2025-12-21 07:00:00,0.2812,-0.363687,0.0195,2025-12-21
1075,2025-12-21 08:00:00,0.2368,-0.608720,1.1077,2025-12-21
1076,2025-12-21 09:00:00,0.2930,-0.298565,-0.1827,2025-12-21


In [3]:
daily = (
    df.groupby("day", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={"day": "Date"})
    .sort_values("Date")
)

daily.to_csv(OUTPUT_PATH, index=False)

print(f"Rows after cleaning: {len(daily)}")
print(f"Saved cleaned file to: {OUTPUT_PATH.resolve()}")
daily.head()

Rows after cleaning: 1079
Saved cleaned file to: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\equity_data\gdelt_data_daily.csv


,Date,gdelt_articles,gdelt_robust,sentiment_score
0,2023-01-01,0.3065,-0.224062,-0.9249
1,2023-01-02,0.4465,0.548565,-0.7794
2,2023-01-03,0.8445,2.745033,-0.9603
3,2023-01-04,0.8318,2.674945,-1.7953
4,2023-01-05,0.5053,0.873068,-1.3037


In [4]:
FULL_OUTPUT_PATH = Path("../data/equity_data/gdelt_data_daily_filled.csv")

full_range = pd.date_range(daily["Date"].min(), daily["Date"].max(), freq="D")
daily_full = daily.set_index("Date").reindex(full_range)
daily_full.index.name = "Date"

# For missing days this interpolates between the previous and next available values.
daily_full = daily_full.interpolate(method="time", limit_direction="both")
daily_full = daily_full.reset_index()

daily_full.to_csv(FULL_OUTPUT_PATH, index=False)

print(f"Rows after filling missing days: {len(daily_full)}")
print(f"Saved filled file to: {FULL_OUTPUT_PATH.resolve()}")
daily_full.head()

Rows after filling missing days: 1097
Saved filled file to: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\data\equity_data\gdelt_data_daily_filled.csv


,Date,gdelt_articles,gdelt_robust,sentiment_score
0,2023-01-01,0.3065,-0.224062,-0.9249
1,2023-01-02,0.4465,0.548565,-0.7794
2,2023-01-03,0.8445,2.745033,-0.9603
3,2023-01-04,0.8318,2.674945,-1.7953
4,2023-01-05,0.5053,0.873068,-1.3037


NameError: name 'size' is not defined